# Chapter 7.5: Tool-Augmented Rec Agents

## Learning Objectives

By the end of this notebook, you will be able to:

1. Design API-calling mechanisms for real-time information retrieval in recommendations
2. Implement retrieval-augmented recommendation with search integration
3. Build knowledge retrieval tools for product databases and external sources
4. Create calculator tools for price comparison and budget optimization
5. Implement intelligent tool selection and routing for rec agents
6. Evaluate tool-augmented agents against baselines
7. Handle tool failures gracefully in production settings

## Prerequisites

- Chapter 7.3 (LLM Agents for Recommendation)
- Understanding of search and retrieval systems
- Basic API design concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part7/chapter_7.5_tool_augmented.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part7/chapter_7.5_tool_augmented.ipynb)

In [ ]:
import numpy as np
import random
import json
import re
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple, Any, Callable
from dataclasses import dataclass, field
from collections import defaultdict

np.random.seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8')
print("All imports successful.")

## 1. Why Tool-Augmented Recommendation?

Traditional rec systems rely on static features. Tool-augmented agents can:

1. **Access real-time data**: Current prices, stock availability, reviews
2. **Perform complex reasoning**: Budget calculations, feature comparisons
3. **Retrieve external knowledge**: Product specs, expert reviews, safety info
4. **Verify facts**: Check claims, validate compatibility

> **💡 Concept:** Tools bridge the gap between the agent's parametric knowledge (what it learned during training) and the non-parametric knowledge (real-time, factual data) needed for accurate recommendations. This is related to **Retrieval-Augmented Generation (RAG)** (Lewis et al., 2020).

### Tool Taxonomy for Recommendation

| Tool Type | Purpose | Example |
|-----------|---------|--------|
| Search | Find relevant items | Keyword/semantic search |
| Filter | Narrow candidates | Price range, category, rating |
| Knowledge | Retrieve details | Product specs, reviews |
| Calculator | Numeric reasoning | Price comparison, budget |
| Validator | Check constraints | Compatibility, availability |

In [ ]:
# Synthetic product database
@dataclass
class Product:
    product_id: int
    name: str
    category: str
    price: float
    original_price: float
    rating: float
    n_reviews: int
    in_stock: bool
    features: Dict[str, str]
    tags: List[str]
    brand: str
    compatible_with: List[int] = field(default_factory=list)


def create_product_database(n_products: int = 80) -> List[Product]:
    rng = np.random.RandomState(42)
    
    categories = {
        'Laptop': {'brands': ['TechPro', 'ByteBook', 'CompuMax'], 
                   'features': {'RAM': ['8GB', '16GB', '32GB'], 'Storage': ['256GB', '512GB', '1TB'],
                               'Display': ['13"', '14"', '15.6"', '16"']}},
        'Headphones': {'brands': ['SoundMax', 'AudioPure', 'BeatWave'],
                       'features': {'Type': ['Over-ear', 'In-ear', 'On-ear'], 
                                   'Connectivity': ['Bluetooth 5.0', 'Wired', 'Bluetooth 5.3'],
                                   'ANC': ['Yes', 'No']}},
        'Smartphone': {'brands': ['PhoneX', 'MobiTech', 'CallPro'],
                       'features': {'Screen': ['6.1"', '6.5"', '6.7"'],
                                   'Camera': ['12MP', '48MP', '108MP'],
                                   'Battery': ['4000mAh', '5000mAh', '6000mAh']}},
        'Tablet': {'brands': ['TabPro', 'PadMax', 'SlateBook'],
                   'features': {'Screen': ['10.2"', '11"', '12.9"'],
                                'Stylus': ['Included', 'Sold separately', 'Not supported']}},
    }
    
    all_tags = ['budget', 'premium', 'portable', 'powerful', 'lightweight',
                'noise-cancelling', 'wireless', 'high-res', 'gaming', 'professional']
    
    products = []
    for i in range(n_products):
        cat_name = list(categories.keys())[i % len(categories)]
        cat = categories[cat_name]
        brand = rng.choice(cat['brands'])
        
        features = {}
        for feat_name, feat_options in cat['features'].items():
            features[feat_name] = rng.choice(feat_options)
        
        original_price = round(rng.uniform(50, 2000), 2)
        discount = rng.choice([0, 0, 0, 0.1, 0.15, 0.2, 0.3])
        price = round(original_price * (1 - discount), 2)
        
        n_tags = rng.randint(2, 5)
        tags = list(rng.choice(all_tags, n_tags, replace=False))
        
        products.append(Product(
            product_id=i,
            name=f"{brand} {cat_name} {i}",
            category=cat_name,
            price=price,
            original_price=original_price,
            rating=round(rng.uniform(2.5, 5.0), 1),
            n_reviews=int(rng.exponential(50)),
            in_stock=rng.random() > 0.15,
            features=features,
            tags=tags,
            brand=brand,
            compatible_with=list(rng.choice(n_products, rng.randint(0, 5), replace=False))
        ))
    
    return products


products = create_product_database(80)
print(f"Database: {len(products)} products")
print(f"Categories: {set(p.category for p in products)}")
print(f"\nSample: {products[0].name}, ${products[0].price}, {products[0].features}")

## 2. Building the Tool Suite

We implement a comprehensive set of tools for product recommendation.

In [ ]:
class ToolResult:
    """Standardized tool result."""
    def __init__(self, success: bool, data: Any, message: str = ""):
        self.success = success
        self.data = data
        self.message = message
    
    def __repr__(self):
        return f"ToolResult(success={self.success}, message='{self.message}', data_len={len(str(self.data))})"


class ProductSearchTool:
    """Semantic and keyword search over products."""
    
    def __init__(self, products: List[Product]):
        self.products = products
        self.name = "product_search"
        self.description = "Search products by keywords, category, tags, or brand"
        # Build simple inverted index
        self.index = defaultdict(set)
        for p in products:
            for word in p.name.lower().split():
                self.index[word].add(p.product_id)
            self.index[p.category.lower()].add(p.product_id)
            self.index[p.brand.lower()].add(p.product_id)
            for tag in p.tags:
                self.index[tag.lower()].add(p.product_id)
    
    def __call__(self, query: str = "", category: str = "",
                 tags: List[str] = None, brand: str = "",
                 max_results: int = 10) -> ToolResult:
        matching_ids = set(range(len(self.products)))
        
        if category:
            cat_matches = {p.product_id for p in self.products 
                          if p.category.lower() == category.lower()}
            matching_ids &= cat_matches
        
        if brand:
            brand_matches = {p.product_id for p in self.products
                            if p.brand.lower() == brand.lower()}
            matching_ids &= brand_matches
        
        if tags:
            for tag in tags:
                tag_matches = self.index.get(tag.lower(), set())
                if tag_matches:
                    matching_ids &= tag_matches
        
        if query:
            query_matches = set()
            for word in query.lower().split():
                query_matches |= self.index.get(word, set())
            if query_matches:
                matching_ids &= query_matches
        
        results = [self.products[pid] for pid in matching_ids][:max_results]
        
        if not results:
            return ToolResult(True, [], "No products found matching criteria")
        
        return ToolResult(True, results, f"Found {len(results)} products")


class PriceComparisonTool:
    """Compare prices and find deals."""
    
    def __init__(self, products: List[Product]):
        self.products = {p.product_id: p for p in products}
        self.name = "price_comparison"
        self.description = "Compare prices, find discounts, calculate savings"
    
    def __call__(self, product_ids: List[int] = None,
                 budget: float = 0,
                 find_deals: bool = False) -> ToolResult:
        if find_deals:
            deals = []
            for p in self.products.values():
                discount_pct = (p.original_price - p.price) / p.original_price * 100
                if discount_pct > 5:
                    deals.append({
                        'product': p,
                        'original_price': p.original_price,
                        'current_price': p.price,
                        'discount_pct': round(discount_pct, 1),
                        'savings': round(p.original_price - p.price, 2)
                    })
            deals.sort(key=lambda x: -x['discount_pct'])
            return ToolResult(True, deals[:10], f"Found {len(deals)} deals")
        
        if product_ids:
            prods = [self.products[pid] for pid in product_ids if pid in self.products]
            comparison = []
            for p in prods:
                comparison.append({
                    'name': p.name,
                    'price': p.price,
                    'original_price': p.original_price,
                    'savings': round(p.original_price - p.price, 2),
                    'rating': p.rating,
                    'value_score': round(p.rating / (p.price / 100), 2)  # Rating per $100
                })
            comparison.sort(key=lambda x: -x['value_score'])
            
            total = sum(c['price'] for c in comparison)
            msg = f"Total: ${total:.2f}"
            if budget > 0:
                msg += f", Budget: ${budget:.2f}, Remaining: ${budget-total:.2f}"
            
            return ToolResult(True, comparison, msg)
        
        return ToolResult(False, None, "No product IDs or find_deals flag provided")


class ProductDetailsTool:
    """Retrieve detailed product information."""
    
    def __init__(self, products: List[Product]):
        self.products = {p.product_id: p for p in products}
        self.name = "product_details"
        self.description = "Get detailed specs, reviews summary, and compatibility info"
        # Synthetic reviews
        self.reviews = {}
        rng = np.random.RandomState(42)
        sentiments = ['Great product!', 'Good value.', 'Decent quality.', 
                      'Could be better.', 'Not recommended.']
        for p in products:
            n = min(p.n_reviews, 5)
            self.reviews[p.product_id] = [
                {'rating': max(1, min(5, int(p.rating + rng.normal(0, 0.8)))),
                 'text': rng.choice(sentiments)}
                for _ in range(n)
            ]
    
    def __call__(self, product_id: int) -> ToolResult:
        if product_id not in self.products:
            return ToolResult(False, None, f"Product {product_id} not found")
        
        p = self.products[product_id]
        details = {
            'name': p.name,
            'category': p.category,
            'brand': p.brand,
            'price': p.price,
            'original_price': p.original_price,
            'rating': p.rating,
            'n_reviews': p.n_reviews,
            'in_stock': p.in_stock,
            'features': p.features,
            'tags': p.tags,
            'compatible_with': p.compatible_with,
            'top_reviews': self.reviews.get(p.product_id, []),
        }
        return ToolResult(True, details, f"Details for {p.name}")


class CompatibilityChecker:
    """Check product compatibility."""
    
    def __init__(self, products: List[Product]):
        self.products = {p.product_id: p for p in products}
        self.name = "compatibility_check"
        self.description = "Check if two products are compatible with each other"
    
    def __call__(self, product_id_1: int, product_id_2: int) -> ToolResult:
        if product_id_1 not in self.products or product_id_2 not in self.products:
            return ToolResult(False, None, "Product not found")
        
        p1 = self.products[product_id_1]
        p2 = self.products[product_id_2]
        
        compatible = (product_id_2 in p1.compatible_with or 
                     product_id_1 in p2.compatible_with)
        
        return ToolResult(True, {
            'compatible': compatible,
            'product_1': p1.name,
            'product_2': p2.name
        }, f"{'Compatible' if compatible else 'Not compatible'}")


class AvailabilityTool:
    """Check stock availability."""
    
    def __init__(self, products: List[Product]):
        self.products = {p.product_id: p for p in products}
        self.name = "availability_check"
        self.description = "Check if products are in stock"
    
    def __call__(self, product_ids: List[int]) -> ToolResult:
        status = {}
        for pid in product_ids:
            if pid in self.products:
                p = self.products[pid]
                status[pid] = {'name': p.name, 'in_stock': p.in_stock}
        
        available = [pid for pid, s in status.items() if s['in_stock']]
        return ToolResult(True, status, f"{len(available)}/{len(status)} in stock")


# Create tool suite
tool_suite = {
    'search': ProductSearchTool(products),
    'price_compare': PriceComparisonTool(products),
    'details': ProductDetailsTool(products),
    'compatibility': CompatibilityChecker(products),
    'availability': AvailabilityTool(products),
}

# Demo tools
print("=== Search Tool ===")
result = tool_suite['search'](category='Laptop', tags=['portable'])
print(f"Result: {result.message}")
for p in result.data[:3]:
    print(f"  {p.name}: ${p.price}, {p.rating}/5.0")

print("\n=== Price Comparison ===")
result = tool_suite['price_compare'](find_deals=True)
print(f"Result: {result.message}")
for deal in result.data[:3]:
    print(f"  {deal['product'].name}: ${deal['current_price']} (was ${deal['original_price']}, save {deal['discount_pct']}%)")

## 3. Tool Selection and Routing

A key challenge is deciding **which tools to call** and **in what order**. This requires:

1. **Intent classification**: What does the user want?
2. **Tool mapping**: Which tools serve this intent?
3. **Execution planning**: What order and with what parameters?

$$P(\text{tool} | \text{query}) = \text{Router}(\text{query}, \text{available\_tools})$$

> **🔑 Pro Tip:** In production, tool selection can be learned from user interaction logs. Start with rule-based routing and gradually replace with learned classifiers.

In [ ]:
class ToolRouter:
    """Routes queries to appropriate tools."""
    
    def __init__(self, tools: Dict[str, Any]):
        self.tools = tools
        self.routing_rules = {
            'search': ['find', 'looking for', 'show me', 'recommend', 'suggest', 'want'],
            'price_compare': ['price', 'compare', 'cost', 'budget', 'cheap', 'expensive', 'deal', 'discount', 'save'],
            'details': ['details', 'specs', 'features', 'review', 'tell me about', 'information'],
            'compatibility': ['compatible', 'work with', 'fit', 'pair with'],
            'availability': ['stock', 'available', 'buy', 'order'],
        }
    
    def route(self, query: str) -> List[Dict[str, Any]]:
        """Determine which tools to call and in what order."""
        query_lower = query.lower()
        tool_scores = {}
        
        for tool_name, keywords in self.routing_rules.items():
            score = sum(1 for kw in keywords if kw in query_lower)
            if score > 0:
                tool_scores[tool_name] = score
        
        if not tool_scores:
            tool_scores['search'] = 1  # Default to search
        
        # Sort by score, build execution plan
        sorted_tools = sorted(tool_scores.items(), key=lambda x: -x[1])
        
        plan = []
        for tool_name, score in sorted_tools:
            params = self._extract_params(query_lower, tool_name)
            plan.append({
                'tool': tool_name,
                'params': params,
                'priority': score
            })
        
        return plan
    
    def _extract_params(self, query: str, tool_name: str) -> Dict:
        """Extract tool parameters from query."""
        params = {}
        
        # Extract category
        for cat in ['laptop', 'headphones', 'smartphone', 'tablet']:
            if cat in query:
                params['category'] = cat.capitalize()
        
        # Extract budget
        budget_match = re.search(r'\$?(\d+)', query)
        if budget_match:
            params['budget'] = float(budget_match.group(1))
        
        # Extract tags
        tag_keywords = ['budget', 'premium', 'portable', 'powerful', 'lightweight',
                       'noise-cancelling', 'wireless', 'gaming', 'professional']
        found_tags = [t for t in tag_keywords if t in query]
        if found_tags:
            params['tags'] = found_tags
        
        return params


router = ToolRouter(tool_suite)

test_queries = [
    "Find me a budget laptop under $500",
    "Compare prices of the top headphones",
    "Is the TechPro Laptop compatible with AudioPure headphones?",
    "Show me the best deals on wireless headphones",
    "I want details and reviews for smartphone 2",
]

for query in test_queries:
    plan = router.route(query)
    print(f"\nQuery: '{query}'")
    for step in plan:
        print(f"  -> {step['tool']} (priority: {step['priority']}) params: {step['params']}")

## 4. Tool-Augmented Recommendation Agent

Now we build the complete agent that combines tool routing, execution, and result synthesis.

In [ ]:
class ToolAugmentedRecAgent:
    """Recommendation agent with tool-calling capabilities."""
    
    def __init__(self, products: List[Product], tools: Dict[str, Any]):
        self.products = products
        self.product_index = {p.product_id: p for p in products}
        self.tools = tools
        self.router = ToolRouter(tools)
        self.conversation_history = []
        self.tool_call_log = []
    
    def recommend(self, query: str, user_prefs: Dict = None,
                  budget: float = 0, top_k: int = 5) -> Dict:
        """Process a recommendation request using tools."""
        self.tool_call_log = []
        
        # Step 1: Route to tools
        plan = self.router.route(query)
        
        # Step 2: Execute tools
        tool_results = []
        candidate_products = set()
        
        for step in plan:
            tool_name = step['tool']
            params = step['params']
            
            result = self._execute_tool(tool_name, params, query)
            if result and result.success:
                tool_results.append({'tool': tool_name, 'result': result})
                
                # Extract product IDs from results
                if isinstance(result.data, list):
                    for item in result.data:
                        if isinstance(item, Product):
                            candidate_products.add(item.product_id)
                        elif isinstance(item, dict) and 'product' in item:
                            candidate_products.add(item['product'].product_id)
        
        # Step 3: If no candidates from tools, fall back to general search
        if not candidate_products:
            result = self.tools['search'](max_results=20)
            if result.success:
                candidate_products = {p.product_id for p in result.data}
        
        # Step 4: Score and rank candidates
        scored = self._score_candidates(candidate_products, user_prefs, budget)
        
        # Step 5: Verify availability of top candidates
        top_ids = [s['product_id'] for s in scored[:top_k * 2]]
        avail = self.tools['availability'](top_ids)
        if avail.success:
            available_ids = {pid for pid, s in avail.data.items() if s['in_stock']}
            scored = [s for s in scored if s['product_id'] in available_ids]
        
        # Step 6: Generate response
        final_recs = scored[:top_k]
        explanations = self._generate_explanations(final_recs, query, user_prefs)
        
        return {
            'recommendations': final_recs,
            'explanations': explanations,
            'tools_used': [tc['tool'] for tc in self.tool_call_log],
            'n_candidates': len(candidate_products),
        }
    
    def _execute_tool(self, tool_name: str, params: Dict, query: str) -> Optional[ToolResult]:
        """Execute a tool with error handling."""
        tool = self.tools.get(tool_name)
        if not tool:
            return None
        
        try:
            if tool_name == 'search':
                result = tool(
                    category=params.get('category', ''),
                    tags=params.get('tags', []),
                    max_results=20
                )
            elif tool_name == 'price_compare':
                result = tool(
                    budget=params.get('budget', 0),
                    find_deals='deal' in query.lower() or 'discount' in query.lower()
                )
            else:
                result = ToolResult(False, None, f"Tool {tool_name} not handled")
            
            self.tool_call_log.append({
                'tool': tool_name,
                'params': params,
                'success': result.success,
                'message': result.message
            })
            return result
        except Exception as e:
            self.tool_call_log.append({
                'tool': tool_name,
                'params': params,
                'success': False,
                'message': str(e)
            })
            return ToolResult(False, None, str(e))
    
    def _score_candidates(self, product_ids: set, user_prefs: Dict = None,
                          budget: float = 0) -> List[Dict]:
        """Score and rank candidate products."""
        scored = []
        for pid in product_ids:
            p = self.product_index.get(pid)
            if not p:
                continue
            
            score = 0.0
            
            # Rating component
            score += (p.rating / 5.0) * 0.3
            
            # Review count (popularity signal)
            score += min(p.n_reviews / 100, 1.0) * 0.1
            
            # Discount bonus
            if p.price < p.original_price:
                discount = (p.original_price - p.price) / p.original_price
                score += discount * 0.15
            
            # Preference matching
            if user_prefs:
                pref_tags = user_prefs.get('tags', [])
                pref_cats = user_prefs.get('categories', [])
                tag_overlap = len(set(pref_tags) & set(p.tags))
                score += tag_overlap * 0.15
                if p.category in pref_cats:
                    score += 0.2
            
            # Budget fit
            if budget > 0:
                if p.price <= budget:
                    score += 0.1
                else:
                    score -= 0.2  # Penalty for over budget
            
            scored.append({
                'product_id': pid,
                'product': p,
                'score': score
            })
        
        scored.sort(key=lambda x: -x['score'])
        return scored
    
    def _generate_explanations(self, recs: List[Dict], query: str,
                               user_prefs: Dict = None) -> List[str]:
        """Generate explanations for each recommendation."""
        explanations = []
        for rec in recs:
            p = rec['product']
            parts = []
            
            if p.rating >= 4.5:
                parts.append(f"Highly rated at {p.rating}/5.0 ({p.n_reviews} reviews)")
            elif p.rating >= 4.0:
                parts.append(f"Well reviewed at {p.rating}/5.0")
            
            if p.price < p.original_price:
                savings = round(p.original_price - p.price, 2)
                parts.append(f"On sale - save ${savings}")
            
            if user_prefs and user_prefs.get('tags'):
                matching = set(user_prefs['tags']) & set(p.tags)
                if matching:
                    parts.append(f"Matches your preference for {', '.join(matching)}")
            
            feat_strs = [f"{k}: {v}" for k, v in list(p.features.items())[:2]]
            if feat_strs:
                parts.append(f"Features: {', '.join(feat_strs)}")
            
            explanations.append(". ".join(parts) if parts else f"A solid {p.category} option.")
        
        return explanations


# Create and test agent
agent = ToolAugmentedRecAgent(products, tool_suite)

user_prefs = {'tags': ['portable', 'wireless'], 'categories': ['Headphones', 'Laptop']}

queries = [
    ("Find me portable wireless headphones under $200", 200),
    ("Show me the best laptop deals", 1000),
    ("I want a budget smartphone", 300),
]

for query, budget in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    result = agent.recommend(query, user_prefs=user_prefs, budget=budget, top_k=3)
    
    print(f"Tools used: {result['tools_used']}")
    print(f"Candidates evaluated: {result['n_candidates']}")
    print(f"\nRecommendations:")
    for i, (rec, expl) in enumerate(zip(result['recommendations'], result['explanations']), 1):
        p = rec['product']
        print(f"  {i}. {p.name} - ${p.price} (score: {rec['score']:.3f})")
        print(f"     {expl}")

## 5. Evaluating Tool Usage Effectiveness

We compare the tool-augmented agent against a baseline that doesn't use tools.

In [ ]:
class BaselineRecAgent:
    """Simple recommendation agent without tools."""
    
    def __init__(self, products: List[Product]):
        self.products = products
    
    def recommend(self, query: str, user_prefs: Dict = None,
                  budget: float = 0, top_k: int = 5) -> Dict:
        # Simple scoring without tools
        scored = []
        for p in self.products:
            score = p.rating / 5.0
            if user_prefs and user_prefs.get('tags'):
                overlap = len(set(user_prefs['tags']) & set(p.tags))
                score += overlap * 0.2
            if budget > 0 and p.price <= budget:
                score += 0.1
            scored.append({'product_id': p.product_id, 'product': p, 'score': score})
        
        scored.sort(key=lambda x: -x['score'])
        final = scored[:top_k]
        return {
            'recommendations': final,
            'explanations': [f"{r['product'].name}" for r in final],
            'tools_used': [],
            'n_candidates': len(self.products)
        }


def evaluate_agents(agents: Dict[str, Any], test_queries: List[Dict],
                    products: List[Product]):
    """Compare multiple agents on test queries."""
    results = defaultdict(lambda: defaultdict(list))
    
    for query_data in test_queries:
        query = query_data['query']
        budget = query_data.get('budget', 0)
        prefs = query_data.get('prefs', None)
        expected_category = query_data.get('expected_category', None)
        
        for agent_name, agent in agents.items():
            result = agent.recommend(query, user_prefs=prefs, budget=budget, top_k=5)
            recs = result['recommendations']
            
            if not recs:
                continue
            
            # Relevance: category match
            if expected_category:
                cat_match = np.mean([1 if r['product'].category == expected_category else 0 
                                    for r in recs])
                results[agent_name]['category_relevance'].append(cat_match)
            
            # Budget compliance
            if budget > 0:
                in_budget = np.mean([1 if r['product'].price <= budget else 0 for r in recs])
                results[agent_name]['budget_compliance'].append(in_budget)
            
            # Average rating of recommended items
            avg_rating = np.mean([r['product'].rating for r in recs])
            results[agent_name]['avg_rating'].append(avg_rating)
            
            # Availability
            in_stock = np.mean([1 if r['product'].in_stock else 0 for r in recs])
            results[agent_name]['availability'].append(in_stock)
            
            # Tool efficiency
            results[agent_name]['n_tools'].append(len(result['tools_used']))
    
    return {agent: {m: np.mean(v) for m, v in metrics.items()} 
            for agent, metrics in results.items()}


# Test queries
test_queries = [
    {'query': 'Find portable laptop under $800', 'budget': 800, 
     'prefs': {'tags': ['portable'], 'categories': ['Laptop']}, 'expected_category': 'Laptop'},
    {'query': 'Wireless headphones with noise cancelling', 'budget': 300,
     'prefs': {'tags': ['wireless', 'noise-cancelling'], 'categories': ['Headphones']}, 'expected_category': 'Headphones'},
    {'query': 'Budget smartphone for gaming', 'budget': 400,
     'prefs': {'tags': ['budget', 'gaming'], 'categories': ['Smartphone']}, 'expected_category': 'Smartphone'},
    {'query': 'Premium tablet for professional work', 'budget': 1500,
     'prefs': {'tags': ['premium', 'professional'], 'categories': ['Tablet']}, 'expected_category': 'Tablet'},
    {'query': 'Show me the best deals on headphones', 'budget': 200,
     'prefs': {'tags': [], 'categories': ['Headphones']}, 'expected_category': 'Headphones'},
]

agents = {
    'Baseline': BaselineRecAgent(products),
    'Tool-Augmented': ToolAugmentedRecAgent(products, tool_suite),
}

eval_results = evaluate_agents(agents, test_queries, products)

print("\nEvaluation Results:")
metrics = ['category_relevance', 'budget_compliance', 'avg_rating', 'availability']
print(f"{'Metric':<25} {'Baseline':<15} {'Tool-Augmented':<15}")
print("-" * 55)
for m in metrics:
    baseline_val = eval_results.get('Baseline', {}).get(m, 0)
    tool_val = eval_results.get('Tool-Augmented', {}).get(m, 0)
    print(f"{m:<25} {baseline_val:<15.3f} {tool_val:<15.3f}")

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics))
width = 0.35

baseline_vals = [eval_results.get('Baseline', {}).get(m, 0) for m in metrics]
tool_vals = [eval_results.get('Tool-Augmented', {}).get(m, 0) for m in metrics]

ax.bar(x - width/2, baseline_vals, width, label='Baseline', color='steelblue')
ax.bar(x + width/2, tool_vals, width, label='Tool-Augmented', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics], rotation=15)
ax.set_ylabel('Score')
ax.set_title('Tool-Augmented vs Baseline Agent')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

## 6. Handling Tool Failures

In production, tools can fail (API timeouts, empty results, errors). Robust agents need fallback strategies.

> **⚠️ Common Pitfall:** A tool failure should never cause the entire recommendation to fail. Always have fallback paths that degrade gracefully.

In [ ]:
class RobustToolExecutor:
    """Executes tools with retry, timeout, and fallback."""
    
    def __init__(self, tools: Dict[str, Any], max_retries: int = 2):
        self.tools = tools
        self.max_retries = max_retries
        self.failure_counts = defaultdict(int)
        self.success_counts = defaultdict(int)
    
    def execute(self, tool_name: str, **kwargs) -> ToolResult:
        """Execute with retries and fallback."""
        tool = self.tools.get(tool_name)
        if not tool:
            return ToolResult(False, None, f"Tool '{tool_name}' not found")
        
        for attempt in range(self.max_retries + 1):
            try:
                # Simulate random failures (10% chance)
                if np.random.random() < 0.1:
                    raise ConnectionError("Simulated API timeout")
                
                result = tool(**kwargs)
                self.success_counts[tool_name] += 1
                return result
            except Exception as e:
                self.failure_counts[tool_name] += 1
                if attempt < self.max_retries:
                    continue  # Retry
                else:
                    return ToolResult(False, None, f"Failed after {self.max_retries+1} attempts: {e}")
    
    def get_reliability_stats(self) -> Dict:
        stats = {}
        for tool_name in set(list(self.success_counts.keys()) + list(self.failure_counts.keys())):
            s = self.success_counts[tool_name]
            f = self.failure_counts[tool_name]
            stats[tool_name] = {
                'successes': s,
                'failures': f,
                'reliability': s / max(s + f, 1)
            }
        return stats


# Simulate many tool calls
executor = RobustToolExecutor(tool_suite)

for _ in range(100):
    executor.execute('search', category='Laptop', max_results=5)
    executor.execute('price_compare', find_deals=True)
    executor.execute('availability', product_ids=[0, 1, 2, 3])

stats = executor.get_reliability_stats()
print("Tool Reliability Stats:")
for tool, s in stats.items():
    print(f"  {tool}: {s['reliability']:.1%} reliability ({s['successes']} ok, {s['failures']} fail)")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
tools_names = list(stats.keys())
reliabilities = [stats[t]['reliability'] for t in tools_names]
ax.barh(tools_names, reliabilities, color='steelblue')
ax.set_xlabel('Reliability')
ax.set_title('Tool Reliability After 100 Calls Each')
ax.set_xlim(0, 1.05)
for i, v in enumerate(reliabilities):
    ax.text(v + 0.01, i, f'{v:.1%}', va='center')
plt.tight_layout()
plt.show()

## 7. Summary

| Tool Type | When to Use | Key Benefit |
|-----------|------------|-------------|
| Search | Finding items matching criteria | Precise retrieval |
| Price Comparison | Budget-aware recommendations | Financial reasoning |
| Product Details | Deep-dive requests | Comprehensive info |
| Compatibility | Cross-sell, bundles | Avoid bad pairings |
| Availability | Final recommendation | No out-of-stock recs |

**Key references:**
- Lewis et al. (2020) - RAG: Retrieval-Augmented Generation (Meta)
- Schick et al. (2023) - Toolformer: Language models that use tools
- Huang et al. (2023) - InteRecAgent with tool calling
- Qin et al. (2023) - ToolLLM: Learning to use tools

---

## Exercises

### 🏋️ Exercise 1: Build a Tool-Augmented Rec Agent with Search and Calculator

In [ ]:
# 🏋️ Exercise 1: Enhanced tool-augmented agent

class BundleRecommender:
    """Recommends product bundles within a budget using tools."""
    
    def __init__(self, products: List[Product], tools: Dict):
        self.products = products
        self.tools = tools
    
    def recommend_bundle(self, query: str, budget: float, 
                         max_items: int = 3) -> Dict:
        # TODO:
        # 1. Use search tool to find items matching query
        # 2. Use price comparison to find best value items
        # 3. Use compatibility checker to ensure items work together
        # 4. Use calculator to optimize bundle within budget
        # 5. Return bundle with total price and savings
        pass

# TODO: Test with "I need a laptop, headphones, and a tablet for under $2000"

print("Exercise 1: Implement BundleRecommender")

### 🏋️ Exercise 2: Learned Tool Router

In [ ]:
# 🏋️ Exercise 2: Learn tool routing from data

import torch
import torch.nn as nn

class LearnedToolRouter:
    """Neural network-based tool router."""
    
    def __init__(self, vocab_size: int = 100, embed_dim: int = 16, n_tools: int = 5):
        # TODO: Build a simple text classifier that maps query -> tool probabilities
        # Use bag-of-words encoding + MLP
        self.vocab = {}  # word -> index
        self.model = None  # nn.Module
    
    def build_vocab(self, queries: List[str]):
        # TODO: Build vocabulary from training queries
        pass
    
    def encode_query(self, query: str) -> torch.Tensor:
        # TODO: Bag-of-words encoding
        pass
    
    def train(self, queries: List[str], labels: List[List[int]], n_epochs: int = 50):
        # TODO: Train the router on (query, tool_labels) pairs
        pass
    
    def route(self, query: str) -> List[str]:
        # TODO: Return ordered list of tools to call
        pass

print("Exercise 2: Implement LearnedToolRouter")

### 🏋️ Exercise 3: Tool Result Caching

In [ ]:
# 🏋️ Exercise 3: Implement tool result caching

class CachedToolExecutor:
    """Caches tool results to avoid redundant calls."""
    
    def __init__(self, tools: Dict, cache_size: int = 100, ttl: int = 300):
        self.tools = tools
        self.cache_size = cache_size
        self.ttl = ttl  # Time-to-live in seconds
        self.cache = {}  # key -> (result, timestamp)
        self.hit_count = 0
        self.miss_count = 0
    
    def execute(self, tool_name: str, **kwargs) -> ToolResult:
        # TODO:
        # 1. Create a cache key from tool_name + kwargs
        # 2. Check cache; if hit and not expired, return cached result
        # 3. If miss, execute tool, cache result, return
        # 4. Evict oldest entries when cache is full
        pass
    
    def get_stats(self) -> Dict:
        # TODO: Return hit rate, miss rate, cache size
        pass

print("Exercise 3: Implement CachedToolExecutor")